# Results for model: deepseek-ai/DeepSeek-R1-Distill-Qwen-32B

In [ ]:
import os
import polars as pl
import xgboost as xgb
import numpy as np

# Load data
train_path = './jane-street-real-time-market-data-forecasting/train.parquet'
df = pl.read_parquet(train_path)

# Feature engineering: Calculate TOP_QUANTILE feature
# Group by date_id and calculate top 15% feature_00
quantile_df = (
    df
    .groupby('date_id')
    .agg(
        [
            pl.col('feature_00').sort(reverse=True).alias('sorted_feature_00'),
            pl.count('feature_00').alias('count')
        ]
    )
    .with_columns(
        [
            pl.col('sorted_feature_00').list.get(pl.col('count') * 0.15).alias('top_quantile_value'),
            pl.col('sorted_feature_00').list.slice(0, pl.col('count') * 0.15).alias('top_quantile_list')
        ]
    )
    .explode('top_quantile_list')
    .with_columns(
        [
            (pl.col('feature_00') >= pl.col('top_quantile_value')).alias('is_top_quantile')
        ]
    )
)

# Join the engineered feature back to the main dataframe
df = df.join(quantile_df, on=['date_id', 'feature_00'], how='left')

# Prepare data for training
X = df.drop(['date_id', 'responder_6']).to_numpy()
y = df['responder_6'].to_numpy()

# Train XGBoost model
params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 1,
    'gamma': 0,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_estimators': 100,
    'seed': 42
}

model = xgb.XGBRegressor(**params)
model.fit(X, y)

if __name__ == "__main__":
    pass